In [1]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = '../../infra/data/db/fotmob.db'

conn = sqlite3.connect(DB_PATH)

match_stats = pd.read_sql_query('SELECT * FROM match_stats', conn)
penalties = pd.read_sql_query('SELECT * FROM penalties', conn)

conn.close()

print(match_stats.shape)
print(penalties.shape)

(6035, 112)
(6035, 10)


## Approach

Binary target, per **team-match** row: did this team get awarded a penalty (scored or missed) in this match?

Features are **exponentially weighted moving averages (EMAs)** of each team's pre-match stats:
- Weighting is by *calendar days* since each prior match (`exp(-decay_rate * days_since)`), not by game order — a 3-week gap (international break, postponement) decays more than a normal 4-7 day gap. `decay_rate = 0.0077`/day (~90-day half-life) reused from the project's existing recency-weighting convention (`param_optimisation.py`).
- `window` caps how many of the team's most recent matches are eligible to contribute (10 / 15 / 20 / 25 / 30) — within that cap, the day-decay still governs actual weight.
- Only matches *strictly before* the target match are used — no leakage.
- Teams need at least 3 prior matches for an EMA to be computed.

Two feature sets per stat: `own_*` (the team's own trailing stat) and `opp_*` (the opponent's trailing stat), since winning a penalty also depends on what the opponent tends to do (concede fouls, lose duels, etc.).

**`is_home` and `league_id` (one-hot)** are included as static, non-EMA features — penalty rates differ a lot by competition (Championship 17.0% vs Premier League 23.1% vs Superligaen 26.2% of matches have at least one penalty), so league is a cheap, high-value signal that was missing from the first pass.

**Promotion/relegation EMA swap:** a team's trailing form means something different depending on the level it was earned at — a promoted team's strong Championship-level stats don't represent what to expect of them in the Premier League. So for a team's first matches after being promoted or relegated, its own pre-transition history is **replaced** with its counterpart's most recent history *at the level the team is now entering* — matched by final league position: the Championship's 1st-placed promoted team is seeded with the Premier League's 18th-placed (best of the) relegated team's PL form, 2nd with 19th, 3rd (however it got promoted — automatic or playoff) with 20th, and vice versa for the relegated teams entering the Championship. This uses the existing rolling-window EMA mechanism unchanged — the borrowed history is just older rows with real dates, so it naturally decays away and gets crowded out by the team's own accumulating matches in the new league, exactly like normal EMA history would.

All three leagues (Premier League, Championship, Superligaen) are combined for sample size — penalties are rare (~10.6% of team-match rows, ~19.9% of matches have at least one).

In [2]:
# Candidate stat columns (base name, i.e. without the home_/away_ prefix).
# Excludes: raw list/nested-text fields that are always null (home_shots, home_defense,
# home_duels, home_discipline) since their sub-values already live in dedicated columns,
# and physical_metrics_* which FotMob only started reporting this season (2026-2027) —
# far too sparse to use as a trailing-history feature across 6 seasons of data.
STAT_COLS = [
    'BallPossesion', 'expected_goals', 'total_shots', 'ShotsOnTarget',
    'big_chance', 'big_chance_missed_title', 'accurate_passes', 'accurate_passes_pct',
    'fouls', 'corners', 'ShotsOffTarget', 'blocked_shots', 'shots_woodwork',
    'shots_inside_box', 'shots_outside_box', 'expected_goals_open_play',
    'expected_goals_set_play', 'expected_goals_non_penalty', 'expected_goals_on_target',
    'passes', 'own_half_passes', 'opposition_half_passes', 'long_balls_accurate',
    'long_balls_accurate_pct', 'accurate_crosses', 'accurate_crosses_pct',
    'player_throws', 'touches_opp_box', 'Offsides', 'tackles', 'interceptions',
    'shot_blocks', 'clearances', 'keeper_saves', 'duel_won', 'ground_duels_won',
    'ground_duels_won_pct', 'aerials_won', 'aerials_won_pct', 'dribbles_succeeded',
    'dribbles_succeeded_pct', 'yellow_cards', 'red_cards',
]

ms = match_stats.rename(columns={
    'home_matchstats.headers.tackles': 'home_tackles',
    'away_matchstats.headers.tackles': 'away_tackles',
})
for base in STAT_COLS:
    for side in ('home', 'away'):
        col = f'{side}_{base}'
        if col in ms.columns:
            ms[col] = pd.to_numeric(ms[col], errors='coerce')

ms['match_date'] = pd.to_datetime(ms['match_date'])

pen = penalties[['match_id', 'home_pens', 'away_pens']]
ms = ms.merge(pen, on='match_id', how='left')
ms[['home_pens', 'away_pens']] = ms[['home_pens', 'away_pens']].fillna(0)

ms.shape

(6035, 114)

In [3]:
# Long panel: one row per (team, match).
home_rows = ms[['match_id', 'match_date', 'league_id', 'season', 'home_team', 'away_team', 'home_pens']].copy()
home_rows = home_rows.rename(columns={'home_team': 'team_id', 'away_team': 'opponent_id', 'home_pens': 'pens_awarded'})
home_rows['is_home'] = 1
for base in STAT_COLS:
    home_rows[base] = ms[f'home_{base}']

away_rows = ms[['match_id', 'match_date', 'league_id', 'season', 'away_team', 'home_team', 'away_pens']].copy()
away_rows = away_rows.rename(columns={'away_team': 'team_id', 'home_team': 'opponent_id', 'away_pens': 'pens_awarded'})
away_rows['is_home'] = 0
for base in STAT_COLS:
    away_rows[base] = ms[f'away_{base}']

panel = pd.concat([home_rows, away_rows], ignore_index=True)
panel['target'] = (panel['pens_awarded'] > 0).astype(int)
panel = panel.sort_values(['team_id', 'match_date']).reset_index(drop=True)

print('panel shape:', panel.shape)
print('target rate (awarded a penalty):', panel['target'].mean().round(4))
panel.head()

panel shape: (12070, 52)
target rate (awarded a penalty): 0.106


,match_id,match_date,league_id,season,team_id,opponent_id,pens_awarded,is_home,BallPossesion,expected_goals,...,duel_won,ground_duels_won,ground_duels_won_pct,aerials_won,aerials_won_pct,dribbles_succeeded,dribbles_succeeded_pct,yellow_cards,red_cards,target
0,4494869,2024-07-19,Superligaen,2024-2025,8071,8113,1,1,49,2.76,...,54,34,47,20,53,11,65,3,0,1
1,4494879,2024-07-28,Superligaen,2024-2025,8071,8391,0,0,51,1.19,...,40,27,51,13,46,5,45,1,0,0
2,4494882,2024-08-02,Superligaen,2024-2025,8071,8487,0,1,61,1.92,...,56,41,52,15,45,7,78,1,0,0
3,4494890,2024-08-11,Superligaen,2024-2025,8071,8595,0,0,49,1.32,...,49,36,67,13,39,4,44,0,0,0
4,4494898,2024-08-19,Superligaen,2024-2025,8071,8231,0,1,66,2.97,...,32,22,45,10,36,7,47,0,0,0


## Promotion/relegation swap map

Final league standings (points, then goal difference, then goals for) are computed directly from match results for the Championship and Premier League each season, so we can identify exactly who went up and down and in what order — including edge cases like a play-off winner who finished 4th on points (e.g. Nottingham Forest, 2021-2022) but still counts as the "3rd promoted" team since what matters is their rank *among the promoted teams*, not their absolute table position.

In [4]:
np_matches = pd.read_sql_query(
    "SELECT * FROM np_matches WHERE league_id IN ('Premier_League', 'Championship')",
    sqlite3.connect(DB_PATH),
)
team_names = pd.read_sql_query('SELECT * FROM team_id_mapping', sqlite3.connect(DB_PATH))

def season_standings(league, season):
    """Final table (points, then GD, then GF) for one league-season, from match results."""
    sub = np_matches[(np_matches.league_id == league) & (np_matches.season == season)]
    teams = pd.unique(sub[['home_team', 'away_team']].values.ravel())
    rows = []
    for t in teams:
        home = sub[sub.home_team == t]
        away = sub[sub.away_team == t]
        pts = (
            (home.home_goals > home.away_goals).sum() * 3 + (home.home_goals == home.away_goals).sum()
            + (away.away_goals > away.home_goals).sum() * 3 + (away.away_goals == away.home_goals).sum()
        )
        gf = home.home_goals.sum() + away.away_goals.sum()
        ga = home.away_goals.sum() + away.home_goals.sum()
        rows.append({'team_id': t, 'pts': pts, 'gd': gf - ga, 'gf': gf})
    out = pd.DataFrame(rows).sort_values(['pts', 'gd', 'gf'], ascending=False).reset_index(drop=True)
    out['rank'] = out.index + 1
    return out.set_index('team_id')['rank']

seasons = sorted(panel['season'].unique(), key=lambda s: int(s.split('-')[0]))

# seed_map[(team_id, league_id, entry_season)] = (partner_team_id, partner_league_id, partner_season)
seed_map = {}
for s, s_next in zip(seasons, seasons[1:]):
    champ_s = set(panel[(panel.league_id == 'Championship') & (panel.season == s)]['team_id'])
    pl_s = set(panel[(panel.league_id == 'Premier_League') & (panel.season == s)]['team_id'])
    champ_snext = set(panel[(panel.league_id == 'Championship') & (panel.season == s_next)]['team_id'])
    pl_snext = set(panel[(panel.league_id == 'Premier_League') & (panel.season == s_next)]['team_id'])

    promoted = champ_s & pl_snext
    relegated = pl_s & champ_snext
    if not promoted and not relegated:
        continue

    champ_rank = season_standings('Championship', s)
    pl_rank = season_standings('Premier_League', s)

    promoted_ranked = sorted(promoted, key=lambda t: champ_rank.get(t, 999))
    relegated_ranked = sorted(relegated, key=lambda t: pl_rank.get(t, 999))

    if len(promoted_ranked) != len(relegated_ranked):
        print(f'  note: {s}->{s_next} promoted/relegated count mismatch '
              f'({len(promoted_ranked)} vs {len(relegated_ranked)}) - likely a still-in-progress season; '
              f'unmatched teams get no seed.')

    for p_team, r_team in zip(promoted_ranked, relegated_ranked):
        seed_map[(p_team, 'Premier_League', s_next)] = (r_team, 'Premier_League', s)
        seed_map[(r_team, 'Championship', s_next)] = (p_team, 'Championship', s)

print(f'{len(seed_map)} team-league-season entries have a swap seed')

  note: 2025-2026->2026-2027 promoted/relegated count mismatch (3 vs 1) - likely a still-in-progress season; unmatched teams get no seed.
32 team-league-season entries have a swap seed


In [5]:
WINDOWS = [10, 15, 20, 25, 30]
DECAY_RATE = 0.0077   # ~90-day half-life, matches the project's existing recency-decay convention
MIN_MATCHES = 3        # minimum prior matches required before an EMA is considered valid

def compute_team_emas(seq_df, feature_cols, windows, decay_rate, min_matches, n_real):
    """seq_df is a chronologically sorted sequence ending in the rows we want EMAs for
    (the last n_real rows); anything before that (a swap partner's seed history, if any)
    is phantom context only. Day-decayed, trailing-window-capped, no leakage: row i only
    ever uses rows strictly before it in the sequence.

    NaN-aware per stat column: a handful of stats (touches_opp_box especially, ~10% of
    matches) are missing at the raw match level, e.g. not tracked for some fixtures. Naively
    averaging would let one missing match poison the whole window's weighted mean for that
    stat -- worse for bigger windows since they touch more matches, which was silently
    shrinking the usable sample far more at window 30 than window 10. Instead each column
    masks out its own missing matches and renormalizes weights over what's left."""
    dates = seq_df['match_date'].values.astype('datetime64[D]')
    vals = seq_df[feature_cols].to_numpy(dtype=float)
    n, k = vals.shape
    start = n - n_real
    out = {f'{col}_ema{w}': np.full(n_real, np.nan) for col in feature_cols for w in windows}
    for i in range(start, n):
        for w in windows:
            lo = max(0, i - w)
            if i - lo < min_matches:
                continue
            v = vals[lo:i]                        # (window_len, k)
            days = (dates[i] - dates[lo:i]).astype('timedelta64[D]').astype(float)
            weights = np.exp(-decay_rate * days)   # (window_len,)
            mask = ~np.isnan(v)                    # per-column validity
            w_matrix = weights[:, None] * mask
            wsum = w_matrix.sum(axis=0)
            valid_count = mask.sum(axis=0)
            v_filled = np.where(mask, v, 0.0)
            with np.errstate(invalid='ignore', divide='ignore'):
                wavg = (v_filled * w_matrix).sum(axis=0) / wsum
            wavg = np.where((valid_count >= min_matches) & (wsum > 0), wavg, np.nan)
            for j, col in enumerate(feature_cols):
                out[f'{col}_ema{w}'][i - start] = wavg[j]
    return pd.DataFrame(out, index=seq_df.index[start:])

ema_parts = []
n_seeded_spans = 0
for team_id, tgroup in panel.groupby('team_id'):
    tgroup = tgroup.sort_values('match_date')
    # split into contiguous same-league spans (a league change starts a new span)
    league_change = (tgroup['league_id'] != tgroup['league_id'].shift()).cumsum()
    for _, span in tgroup.groupby(league_change):
        span = span.sort_values('match_date')
        key = (team_id, span['league_id'].iloc[0], span['season'].iloc[0])
        if key in seed_map:
            partner_id, partner_league, partner_season = seed_map[key]
            seed_rows = panel[
                (panel.team_id == partner_id) & (panel.league_id == partner_league) & (panel.season == partner_season)
            ].sort_values('match_date')
            seq = pd.concat([seed_rows, span])
            n_seeded_spans += 1
        else:
            seq = span
        ema_parts.append(compute_team_emas(seq, STAT_COLS, WINDOWS, DECAY_RATE, MIN_MATCHES, n_real=len(span)))

ema_df = pd.concat(ema_parts).sort_index()
print(f'seeded {n_seeded_spans} team-league spans out of', panel.groupby('team_id')['league_id'].apply(lambda s: (s != s.shift()).sum()).sum(), 'total spans')
ema_df.shape

seeded 32 team-league spans out of 105 total spans


(12070, 215)

In [6]:
# Sanity check: Fulham were Championship champions in 2021-2022, promoted to the Premier
# League for 2022-2023, seeded from 18th-placed (best of the relegated) Leeds' PL form —
# NOT from Fulham's own (much stronger, Championship-level) numbers.
fulham_id = team_names.loc[team_names.team_name == 'Fulham', 'team_id'].iloc[0]
leeds_id = team_names.loc[team_names.team_name == 'Leeds', 'team_id'].iloc[0]

fulham_first_pl_idx = panel[
    (panel.team_id == fulham_id) & (panel.league_id == 'Premier_League') & (panel.season == '2022-2023')
].sort_values('match_date').index[0]

print('Fulham entering the PL, own_expected_goals_ema10:', round(ema_df.loc[fulham_first_pl_idx, 'expected_goals_ema10'], 2))
print('  (compare to) Leeds\' last 10 PL 2021-2022 xG values:      ',
      panel.loc[panel[(panel.team_id == leeds_id) & (panel.league_id == 'Premier_League') & (panel.season == '2021-2022')]
                .sort_values('match_date').index[-10:], 'expected_goals'].round(2).tolist())
print('  (NOT Fulham\'s own) last 10 Championship 2021-2022 xG:    ',
      panel.loc[panel[(panel.team_id == fulham_id) & (panel.league_id == 'Championship') & (panel.season == '2021-2022')]
                .sort_values('match_date').index[-10:], 'expected_goals'].round(2).tolist())

Fulham entering the PL, own_expected_goals_ema10: 1.58
  (compare to) Leeds' last 10 PL 2021-2022 xG values:       [2.73, 2.06, 1.18, 0.73, 0.47, 1.3, 0.31, 0.42, 1.67, 1.29]
  (NOT Fulham's own) last 10 Championship 2021-2022 xG:     [0.53, 2.15, 0.95, 3.5, 2.08, 1.46, 0.39, 2.25, 2.73, 0.53]


In [7]:
# Attach each row's own EMAs, then self-join the opponent's EMAs for that same match,
# and one-hot encode league_id as a static (non-EMA) feature alongside is_home.
panel_ema = pd.concat(
    [panel[['match_id', 'match_date', 'league_id', 'season', 'team_id', 'opponent_id', 'is_home', 'target']], ema_df],
    axis=1,
)

ema_cols = list(ema_df.columns)
own = panel_ema.rename(columns={c: f'own_{c}' for c in ema_cols})
opp_lookup = panel_ema[['match_id', 'team_id'] + ema_cols].rename(
    columns={'team_id': 'opponent_id', **{c: f'opp_{c}' for c in ema_cols}}
)

final = own.merge(opp_lookup, on=['match_id', 'opponent_id'], how='left')

league_dummies = pd.get_dummies(final['league_id'], prefix='league')
LEAGUE_COLS = list(league_dummies.columns)
final = pd.concat([final, league_dummies], axis=1)

print('final shape:', final.shape)
final.head()

final shape: (12070, 441)


,match_id,match_date,league_id,season,team_id,opponent_id,is_home,target,own_BallPossesion_ema10,own_BallPossesion_ema15,...,opp_yellow_cards_ema25,opp_yellow_cards_ema30,opp_red_cards_ema10,opp_red_cards_ema15,opp_red_cards_ema20,opp_red_cards_ema25,opp_red_cards_ema30,league_Championship,league_Premier_League,league_Superligaen
0,4494869,2024-07-19,Superligaen,2024-2025,8071,8113,1,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,True
1,4494879,2024-07-28,Superligaen,2024-2025,8071,8391,0,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,True
2,4494882,2024-08-02,Superligaen,2024-2025,8071,8487,1,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,True
3,4494890,2024-08-11,Superligaen,2024-2025,8071,8595,0,0,53.868627,53.868627,...,0.685273,0.685273,0.0,0.0,0.0,0.0,0.0,False,False,True
4,4494898,2024-08-19,Superligaen,2024-2025,8071,8231,1,0,52.541502,52.541502,...,1.453442,1.453442,0.0,0.0,0.0,0.0,0.0,False,False,True


## Correlation matrices per EMA window

For each window, correlate every `own_*`/`opp_*` EMA feature plus the static `is_home`/`league_*` features against the binary target. Rows are dropped per-window wherever a team doesn't yet have enough history for that window's EMA — with the NaN-aware fix above, that's now a near-constant ~11-11.6k rows regardless of window (only window 10 loses a bit more, since it takes the least history to fill).

In [8]:
corr_results = {}
for w in WINDOWS:
    cols = [c for c in final.columns if c.endswith(f'_ema{w}')] + ['is_home'] + LEAGUE_COLS
    sub = final[cols + ['target']].dropna()
    corr_matrix = sub.corr()
    corr_results[w] = corr_matrix['target'].drop('target').sort_values(key=lambda s: s.abs(), ascending=False)
    print(f'=== window {w} (n={len(sub)} rows) ===')
    print(f'  {len(cols)} features, top 15 by |correlation| with target:')
    print(corr_results[w].head(15).to_string())
    print()

=== window 10 (n=11040 rows) ===
  90 features, top 15 by |correlation| with target:
own_opposition_half_passes_ema10        0.077824
own_passes_ema10                        0.074124
own_expected_goals_ema10                0.073997
own_expected_goals_open_play_ema10      0.073946
own_long_balls_accurate_pct_ema10       0.073395
own_accurate_passes_ema10               0.073350
own_touches_opp_box_ema10               0.072184
own_expected_goals_non_penalty_ema10    0.069437
own_expected_goals_on_target_ema10      0.068574
own_big_chance_ema10                    0.063148
own_ShotsOnTarget_ema10                 0.062882
league_Championship                    -0.061120
own_total_shots_ema10                   0.060286
own_shots_inside_box_ema10              0.059156
own_own_half_passes_ema10               0.057085



=== window 15 (n=11610 rows) ===
  90 features, top 15 by |correlation| with target:
own_expected_goals_ema15                0.074218
own_expected_goals_open_play_ema15      0.073700
own_opposition_half_passes_ema15        0.070583
own_touches_opp_box_ema15               0.070112
own_expected_goals_non_penalty_ema15    0.069918
own_long_balls_accurate_pct_ema15       0.069550
own_passes_ema15                        0.068612
own_accurate_passes_ema15               0.067886
own_expected_goals_on_target_ema15      0.065553
own_big_chance_ema15                    0.064512
own_ShotsOnTarget_ema15                 0.062343
league_Championship                    -0.059615
own_total_shots_ema15                   0.059474
own_shots_inside_box_ema15              0.059032
own_own_half_passes_ema15               0.054747



=== window 20 (n=11620 rows) ===
  90 features, top 15 by |correlation| with target:
own_expected_goals_ema20                0.075759
own_expected_goals_open_play_ema20      0.072847
own_expected_goals_non_penalty_ema20    0.071259
own_touches_opp_box_ema20               0.070964
own_opposition_half_passes_ema20        0.070653
own_long_balls_accurate_pct_ema20       0.070089
own_passes_ema20                        0.068914
own_accurate_passes_ema20               0.067899
own_expected_goals_on_target_ema20      0.067693
own_big_chance_ema20                    0.065125
own_ShotsOnTarget_ema20                 0.063545
own_total_shots_ema20                   0.060996
own_shots_inside_box_ema20              0.060405
league_Championship                    -0.059495
own_own_half_passes_ema20               0.054962



=== window 25 (n=11620 rows) ===
  90 features, top 15 by |correlation| with target:
own_expected_goals_ema25                0.076446
own_expected_goals_open_play_ema25      0.073568
own_expected_goals_non_penalty_ema25    0.072104
own_touches_opp_box_ema25               0.072016
own_long_balls_accurate_pct_ema25       0.071000
own_opposition_half_passes_ema25        0.070456
own_expected_goals_on_target_ema25      0.069587
own_passes_ema25                        0.068534
own_accurate_passes_ema25               0.067526
own_ShotsOnTarget_ema25                 0.065763
own_big_chance_ema25                    0.065552
own_total_shots_ema25                   0.062869
own_shots_inside_box_ema25              0.062227
league_Championship                    -0.059495
own_BallPossesion_ema25                 0.054692



=== window 30 (n=11620 rows) ===
  90 features, top 15 by |correlation| with target:
own_expected_goals_ema30                0.076624
own_expected_goals_open_play_ema30      0.073194
own_expected_goals_non_penalty_ema30    0.072428
own_touches_opp_box_ema30               0.072246
own_long_balls_accurate_pct_ema30       0.071635
own_opposition_half_passes_ema30        0.071239
own_expected_goals_on_target_ema30      0.070178
own_passes_ema30                        0.069133
own_accurate_passes_ema30               0.068107
own_ShotsOnTarget_ema30                 0.067095
own_big_chance_ema30                    0.066123
own_total_shots_ema30                   0.064076
own_shots_inside_box_ema30              0.062641
league_Championship                    -0.059495
own_BallPossesion_ema30                 0.055781



In [9]:
# Full feature x feature x target correlation matrix for one window, if you want to
# inspect multicollinearity too (not just the target column) -- e.g. window=15:
w = 15
cols = [c for c in final.columns if c.endswith(f'_ema{w}')] + ['is_home'] + LEAGUE_COLS
full_corr_15 = final[cols + ['target']].dropna().corr()
full_corr_15.shape

(91, 91)

In [10]:
# Consolidated view: same feature's correlation with target across all 5 windows side by side.
# Strips the _emaN suffix so each row is one underlying stat (is_home / league_* have no suffix).
rows = []
for w in WINDOWS:
    s = corr_results[w].copy()
    s.index = [i if not i.endswith(f'_ema{w}') else i.rsplit(f'_ema{w}', 1)[0] for i in s.index]
    s.name = w
    rows.append(s)

summary = pd.concat(rows, axis=1)
summary['mean_abs'] = summary[WINDOWS].abs().mean(axis=1)
summary = summary.sort_values('mean_abs', ascending=False)
summary.head(25)

,10,15,20,25,30,mean_abs
own_expected_goals,0.073997,0.074218,0.075759,0.076446,0.076624,0.075409
own_expected_goals_open_play,0.073946,0.073700,0.072847,0.073568,0.073194,0.073451
own_opposition_half_passes,0.077824,0.070583,0.070653,0.070456,0.071239,0.072151
own_touches_opp_box,0.072184,0.070112,0.070964,0.072016,0.072246,0.071504
own_long_balls_accurate_pct,0.073395,0.069550,0.070089,0.071000,0.071635,0.071134
own_expected_goals_non_penalty,0.069437,0.069918,0.071259,0.072104,0.072428,0.071029
own_passes,0.074124,0.068612,0.068914,0.068534,0.069133,0.069863
own_accurate_passes,0.073350,0.067886,0.067899,0.067526,0.068107,0.068953
own_expected_goals_on_target,0.068574,0.065553,0.067693,0.069587,0.070178,0.068317
own_big_chance,0.063148,0.064512,0.065125,0.065552,0.066123,0.064892


## XGBoost vs. the baseline(s), with walk-forward time-series CV

The correlations above are weak (~0.05-0.07 typically, though `league_Superligaen`/`league_Championship` now show up strongly since league rates differ so much) — so the real question is whether a form-aware, league-aware model beats the flat home/away rate currently feeding the non-penalty Bayes model.

**Why walk-forward, not a single holdout, and not random k-fold:**
- A single fixed train/test split (what the first pass here used — train on everything through 2024-2025, test only on 2025-2026) is high-variance: it tells you how the model did against *one* particular season's idiosyncrasies (that season's referees, that season's competitive balance), not how it generalizes in general. One number, no sense of how noisy it is.
- **Random k-fold is the wrong fix for that**, even though it would give multiple folds. The features here are already leakage-free at construction (each row's EMAs only look backward from that row's own date, regardless of which fold it lands in), so random folds wouldn't leak future *information into a feature* — but rows aren't independent: matches from the same season share latent factors (that season's referee panel, that season's competitive balance, rule/VAR changes) that a random shuffle would let leak between train and test anyway, inflating apparent performance relative to genuinely forecasting an unseen season. It also just doesn't match the deployment reality: this model will only ever be asked to predict matches that haven't happened yet, never to fill in gaps inside an already-mostly-observed season.
- **The actual fix is more folds, still chronological** — walk-forward / expanding-window CV: train on everything up to season *N*, test on season *N+1*, then grow the training window by one season and repeat. Every fold still respects "train strictly precedes test," but instead of one noisy estimate you get several, averaged.

Three models compared per window, per fold:
- `baseline_home_away` — exactly today's approach: flat rate per home/away from the training window.
- `baseline_home_away_league` — same, but split by league too (a free improvement, since league turned out to matter a lot).
- `xgboost` — the swap-seeded EMA features + `is_home` + `league_*`, same shallow/regularized config as before.

In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

XGB_PARAMS = dict(
    n_estimators=300, max_depth=2, learning_rate=0.02,
    subsample=0.7, colsample_bytree=0.5, reg_lambda=5.0, min_child_weight=20,
    eval_metric='logloss', early_stopping_rounds=30, random_state=42,
)

# Expanding-window folds: train grows by one season each time, test is always the next
# unseen season. 2026-2027 is excluded (still in progress, ~29 matches so far).
FOLDS = [
    (['2020-2021', '2021-2022'], ['2022-2023']),
    (['2020-2021', '2021-2022', '2022-2023'], ['2023-2024']),
    (['2020-2021', '2021-2022', '2022-2023', '2023-2024'], ['2024-2025']),
    (['2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025'], ['2025-2026']),
]

cv_results = []
fitted_models = {}  # last (most-data) fold's model per window, for feature importance
for w in WINDOWS:
    ema_cols_w = [c for c in final.columns if c.endswith(f'_ema{w}')]
    cols = ema_cols_w + ['is_home'] + LEAGUE_COLS
    sub = final[cols + ['target', 'league_id', 'season', 'match_date']].dropna().sort_values('match_date')

    for fold_i, (train_seasons, test_seasons) in enumerate(FOLDS):
        train = sub[sub['season'].isin(train_seasons)]
        test = sub[sub['season'].isin(test_seasons)]
        if len(train) < 200 or len(test) < 50:
            continue

        X_train_full, y_train_full = train[cols], train['target']
        X_test, y_test = test[cols], test['target']

        n_val = max(50, int(len(train) * 0.15))
        X_fit, y_fit = X_train_full.iloc[:-n_val], y_train_full.iloc[:-n_val]
        X_val, y_val = X_train_full.iloc[-n_val:], y_train_full.iloc[-n_val:]

        model = XGBClassifier(**XGB_PARAMS)
        model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
        if fold_i == len(FOLDS) - 1:
            fitted_models[w] = model
        xgb_proba = model.predict_proba(X_test)[:, 1]

        home_rate = y_train_full[X_train_full['is_home'] == 1].mean()
        away_rate = y_train_full[X_train_full['is_home'] == 0].mean()
        base1_proba = np.where(X_test['is_home'] == 1, home_rate, away_rate)

        rate_table = train.groupby(['league_id', 'is_home'])['target'].mean()
        base2_proba = np.array([
            rate_table.get((lg, h), y_train_full.mean()) for lg, h in zip(test['league_id'], X_test['is_home'])
        ])

        for label, proba in [
            ('baseline_home_away', base1_proba),
            ('baseline_home_away_league', base2_proba),
            ('xgboost', xgb_proba),
        ]:
            cv_results.append({
                'window': w, 'fold': fold_i, 'test_season': test_seasons[0], 'model': label,
                'n_train': len(X_train_full), 'n_test': len(X_test),
                'log_loss': log_loss(y_test, proba, labels=[0, 1]),
                'brier': brier_score_loss(y_test, proba),
                'auc': roc_auc_score(y_test, proba),
            })

cv_results_df = pd.DataFrame(cv_results)
cv_results_df.head(12)

,window,fold,test_season,model,n_train,n_test,log_loss,brier,auc
0,10,0,2022-2023,baseline_home_away,3214,1570,0.332594,0.092882,0.556477
1,10,0,2022-2023,baseline_home_away_league,3214,1570,0.331079,0.092585,0.576866
2,10,0,2022-2023,xgboost,3214,1570,0.332100,0.092749,0.539324
3,10,1,2023-2024,baseline_home_away,4784,1778,0.332417,0.092941,0.552790
4,10,1,2023-2024,baseline_home_away_league,4784,1778,0.330182,0.092480,0.584082
5,10,1,2023-2024,xgboost,4784,1778,0.329026,0.092226,0.602292
6,10,2,2024-2025,baseline_home_away,6562,2200,0.312285,0.085500,0.555752
7,10,2,2024-2025,baseline_home_away_league,6562,2200,0.311586,0.085422,0.567296
8,10,2,2024-2025,xgboost,6562,2200,0.310965,0.085279,0.586635
9,10,3,2025-2026,baseline_home_away,8762,2232,0.323159,0.089443,0.537515


In [12]:
# Mean across the 4 folds, per window/model -- this is the number that actually matters.
cv_summary = cv_results_df.groupby(['window', 'model'])[['log_loss', 'brier', 'auc']].mean().round(4)
cv_summary

log_loss   brier     auc
window model                                              
10     baseline_home_away           0.3251  0.0902  0.5506
       baseline_home_away_league    0.3233  0.0899  0.5774
       xgboost                      0.3234  0.0899  0.5741
15     baseline_home_away           0.3249  0.0901  0.5503
       baseline_home_away_league    0.3231  0.0898  0.5782
       xgboost                      0.3241  0.0900  0.5689
20     baseline_home_away           0.3246  0.0900  0.5503
       baseline_home_away_league    0.3229  0.0897  0.5781
       xgboost                      0.3229  0.0897  0.5818
25     baseline_home_away           0.3246  0.0900  0.5503
       baseline_home_away_league    0.3229  0.0897  0.5781
       xgboost                      0.3225  0.0896  0.5831
30     baseline_home_away           0.3246  0.0900  0.5503
       baseline_home_away_league    0.3229  0.0897  0.5781
       xgboost                      0.3225  0.0896  0.5782

In [13]:
# Per-fold detail for the best window, to see how consistent the improvement is season-by-season
# rather than just trusting the average.
best_window = cv_summary.xs('xgboost', level='model')['log_loss'].idxmin()
print(f'Best window by mean log-loss: {best_window}')
cv_results_df[cv_results_df.window == best_window].pivot(index='test_season', columns='model', values='log_loss')

Best window by mean log-loss: 25


model,baseline_home_away,baseline_home_away_league,xgboost
test_season,,,
2022-2023,0.330615,0.329147,0.330422
2023-2024,0.332404,0.330391,0.327836
2024-2025,0.312132,0.311298,0.310705
2025-2026,0.323188,0.320593,0.321071


In [14]:
# Feature importance from the best window's final-fold model (trained on the most data: 2020-21..2024-25)
importances = pd.Series(
    fitted_models[best_window].feature_importances_,
    index=[c for c in final.columns if c.endswith(f'_ema{best_window}')] + ['is_home'] + LEAGUE_COLS,
).sort_values(ascending=False)
importances.head(20)

own_long_balls_accurate_pct_ema25     0.023811
own_big_chance_ema25                  0.023199
own_opposition_half_passes_ema25      0.022411
own_expected_goals_on_target_ema25    0.021217
own_accurate_passes_pct_ema25         0.018833
own_expected_goals_open_play_ema25    0.018093
own_passes_ema25                      0.017561
league_Championship                   0.016847
own_own_half_passes_ema25             0.016134
opp_own_half_passes_ema25             0.015127
own_accurate_passes_ema25             0.014832
own_touches_opp_box_ema25             0.014602
is_home                               0.014349
own_clearances_ema25                  0.014122
own_ShotsOffTarget_ema25              0.013965
opp_clearances_ema25                  0.013700
opp_total_shots_ema25                 0.013679
own_ground_duels_won_ema25            0.013581
own_aerials_won_ema25                 0.013324
opp_accurate_passes_ema25             0.013243
dtype: float32

## Head-to-head vs. the *literal* production constants

Everything above compared XGBoost to `baseline_home_away_league`, computed dynamically from each fold's training data — the same shape as what `outputs.ipynb` does (a flat rate per league x home/away), but not the literal frozen numbers currently sitting in the notebooks. That's a fair comparison of *approaches*, but not proof against what's actually deployed today.

So here's the direct version: same walk-forward folds, same window-25 model, but the baseline is the exact hardcoded constants from `premier_league/outputs.ipynb` and `superligaen/outputs.ipynb` (the pre-`*0.78` numbers — that `0.78` converts a penalty-award rate into expected penalty *goals* for the Poisson lambda, a different quantity than the `P(awarded)` this model predicts, so the fair comparison point is the rate *before* that conversion: PL 0.157 home / 0.101 away, Superligaen 0.18 home / 0.13 away). No Championship non-penalty Bayes model exists in the repo, so there's nothing to compare there. Evaluated **per league separately**, not pooled — "does this beat what's running today" is really two separate questions.

In [15]:
# Literal hardcoded production constants (pre-conversion "penalties per game" rate --
# this IS the P(awarded)-equivalent number; the *0.78 in the notebooks converts it to
# expected penalty GOALS for the Poisson lambda, a different quantity than this target).
PROD_RATES = {
    ('Premier_League', 1): 0.157, ('Premier_League', 0): 0.101,
    ('Superligaen', 1): 0.18, ('Superligaen', 0): 0.13,
}

ema_cols_best = [c for c in final.columns if c.endswith(f'_ema{best_window}')]
cols_best = ema_cols_best + ['is_home'] + LEAGUE_COLS
sub_best = final[cols_best + ['target', 'league_id', 'season', 'match_date']].dropna().sort_values('match_date')

prod_results = []
for fold_i, (train_seasons, test_seasons) in enumerate(FOLDS):
    train = sub_best[sub_best['season'].isin(train_seasons)]
    test = sub_best[sub_best['season'].isin(test_seasons)]
    if len(train) < 200 or len(test) < 50:
        continue

    X_train_full, y_train_full = train[cols_best], train['target']
    X_test, y_test = test[cols_best], test['target']
    n_val = max(50, int(len(train) * 0.15))
    X_fit, y_fit = X_train_full.iloc[:-n_val], y_train_full.iloc[:-n_val]
    X_val, y_val = X_train_full.iloc[-n_val:], y_train_full.iloc[-n_val:]

    model = XGBClassifier(**XGB_PARAMS)
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
    xgb_proba_all = model.predict_proba(X_test)[:, 1]

    for lg in ['Premier_League', 'Superligaen']:
        lg_mask = (test['league_id'] == lg).values
        if lg_mask.sum() < 20:
            continue
        y_lg = y_test[lg_mask]
        xgb_lg = xgb_proba_all[lg_mask]
        is_home_lg = X_test['is_home'].values[lg_mask]
        prod_lg = np.array([PROD_RATES[(lg, h)] for h in is_home_lg])

        for label, proba in [('prod_literal', prod_lg), ('xgboost', xgb_lg)]:
            prod_results.append({
                'league': lg, 'test_season': test_seasons[0], 'model': label, 'n_test': lg_mask.sum(),
                'log_loss': log_loss(y_lg, proba, labels=[0, 1]),
                'brier': brier_score_loss(y_lg, proba),
                'auc': roc_auc_score(y_lg, proba) if y_lg.nunique() > 1 else np.nan,
            })

prod_results_df = pd.DataFrame(prod_results)
prod_results_df.groupby(['league', 'model'])[['log_loss', 'brier', 'auc']].mean().round(4)

log_loss   brier     auc
league         model                                 
Premier_League prod_literal    0.3607  0.1036  0.5538
               xgboost         0.3587  0.1031  0.5761
Superligaen    prod_literal    0.4166  0.1254  0.5531
               xgboost         0.4193  0.1260  0.5516

In [16]:
# Per-fold detail, so a single lucky/unlucky season doesn't hide behind the average.
print('Premier League, log-loss by season:')
print(prod_results_df[prod_results_df.league == 'Premier_League'].pivot(index='test_season', columns='model', values='log_loss').round(4))
print()
print('Superligaen, log-loss by season:')
print(prod_results_df[prod_results_df.league == 'Superligaen'].pivot(index='test_season', columns='model', values='log_loss').round(4))

Premier League, log-loss by season:
model        prod_literal  xgboost
test_season                       
2022-2023          0.3747   0.3753
2023-2024          0.3822   0.3790
2024-2025          0.3238   0.3191
2025-2026          0.3621   0.3613

Superligaen, log-loss by season:
model        prod_literal  xgboost
test_season                       
2024-2025          0.4339   0.4390
2025-2026          0.3992   0.3996


### Reading the results

- **The league-aware baseline is a clear, nearly-free win** — splitting the home/away rate by league beats the plain 2-number baseline on log-loss/Brier/AUC in every window/fold, for the cost of a `groupby`. This alone is worth shipping regardless of whether XGBoost gets used at all.
- **XGBoost adds a further, real edge on top of that — most clearly at the longer windows (20, 25, 30), not the shorter ones.** This flipped from an earlier pass of this notebook: a NaN-propagation bug (a handful of stats, `touches_opp_box` especially, are missing on ~10% of matches at the raw level, and one missing match anywhere in the lookback was silently nulling out that whole row for the *entire* window) was shrinking window 30's usable sample much more than window 10's, so it looked artificially worse. With that fixed, all 5 windows have essentially the same sample size, and window 25 wins on log-loss and Brier, **consistently, in all 4 walk-forward folds** — not just on average (see the per-fold table above). Windows 20 and 30 are close behind; window 10 and 15 are now the weaker options, the reverse of the earlier (buggy) result.
- **The promotion/relegation swap mainly helps in the seasons right after a swap boundary** — it's a small slice of any given season's rows (3 teams x 2 directions = 6 teams per year out of ~40+ in the pooled panel), so don't expect it to move the aggregate numbers much by itself; its value is in not actively corrupting those specific teams' early-season predictions with the wrong division's stats, which not having this would otherwise do.
- **Against the literal production constants, the result is more mixed than the pooled numbers suggest — and this is the honest headline, not the pooled walk-forward result above.** For the **Premier League**, XGBoost beats the frozen `0.157`/`0.101` on log-loss in 2 of 4 seasons (clearly in 2024-2025), is essentially tied in the other 2, and wins on average — a real but modest edge, not a rout. For **Superligaen**, XGBoost is actually *worse* than the frozen `0.18`/`0.13` in both available seasons — Superligaen only entered the dataset in 2024-2025, so window-25 EMAs for those teams are built on very little real history (no promotion/relegation seeding exists for Superligaen either, since no second Danish tier is tracked here), and the model has too little to work with yet. **Don't swap in the XGBoost model for Superligaen based on this — the frozen constant is currently the safer bet there.**
- **Net takeaway**: replace the flat 2-number home/away baseline with the league-split version everywhere — that's the highest-value, lowest-effort, lowest-risk change, confirmed against both the pooled and literal comparisons. Layer the window-25 XGBoost model on top **for the Premier League only** for now; for Superligaen, keep the frozen constant (but refresh its number — see the staleness point from earlier in this conversation) until there's more than two seasons of history to train on.

## Other suggestions

- **Referee identity** — available in the raw FotMob JSON (`matchFacts.infoBox.Referee.text`) but not extracted into any DB table yet. Referees are known to vary a lot in penalties-given-per-game, and that's a more direct causal driver than any team stat used here. Would need a small ETL addition, then a referee-level trailing rate with shrinkage toward the league mean (some refs have few games in the sample).
- **Blend rather than fully swap.** Given the model's edge over the league-aware baseline is modest, averaging XGBoost's output with the league-aware baseline (or feeding the baseline rate in as a model feature) is a cheap way to get a floor-not-ceiling result.
- **Shot-location detail from WhoScored** (`match_events` has real x/y and a `shotPenaltyArea` flag) would give a sharper "how often does this team actually get bodies into the box" signal than FotMob's blunter `touches_opp_box`, but it's more work — different match-ID system, needs the team/match mapping to join.
- **More data**: `fotmob_season_downloader.LEAGUES` already has La Liga, Eliteserien, Allsvenskan, and League of Ireland stubbed out (commented out) — turning those on would meaningfully grow N, which directly helps given the signal here is weak-but-real and mostly limited by sample size, not by better features. (Note: La Liga has its own promotion/relegation pairing with Segunda División, not tracked here — the swap logic would need extending to a second promotion/relegation pair if that league gets added.)
- **Calibration matters more than discrimination for feeding into the Bayes model.** A well-calibrated Brier/log-loss model beats one that's merely well-ranked (high AUC) but systematically over- or under-confident — worth a reliability/calibration plot (predicted probability bucket vs. actual rate) before trusting this as an input elsewhere.